In [ ]:
import IPython
display(IPython.display.Javascript('''
(function(){
  if (window.__keepAliveTimer) {
    console.log("[KeepAlive] already running");
    return;
  }
  function SimulateActivity() {
    const timestamp = new Date().toLocaleTimeString();
    document.dispatchEvent(new MouseEvent('mousemove', { bubbles: true }));
    document.dispatchEvent(new KeyboardEvent('keydown', { bubbles: true, key: 'Shift' }));
    console.log("[KeepAlive] activity ping @ " + timestamp);
  }
  window.__keepAliveTimer = setInterval(SimulateActivity, 60000);
  console.log("[KeepAlive] started");
})();
'''))

In [ ]:
import IPython
display(IPython.display.Javascript('''
if (window.__keepAliveTimer) {
  clearInterval(window.__keepAliveTimer);
  window.__keepAliveTimer = null;
  console.log("[KeepAlive] stopped");
} else {
  console.log("[KeepAlive] was not running");
}
'''))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Persiapan Library

In [ ]:
!pip install ultralytics roboflow
import os
from roboflow import Roboflow
from ultralytics import YOLO
from IPython.display import Image, display


# Download Dataset dari Roboflow

In [ ]:
print("Mengunduh dataset dari Roboflow...")

from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
# TODO: replace with your own workspace/project/version.
# Example public chess dataset to fork into your workspace:
# https://universe.roboflow.com/drklc/chess-full-spdfu
project = rf.workspace("YOUR_WORKSPACE_ID").project("YOUR_PROJECT_ID")
version = project.version(1)  # TODO: replace with your dataset version number
dataset = version.download("yolov8")  # detection format (images/ + labels/ + data.yaml), not "folder"

dataset_path = dataset.location


# Mulai Train Model

In [ ]:
print("\n Memulai training model deteksi objek")

model = YOLO("yolo26n.pt")  # detection checkpoint (no "-cls" suffix)

results = model.train(
    data=f"{dataset_path}/data.yaml",  # detection needs the data.yaml, not the raw folder path
    imgsz=640,           # raised from 320: chess pawns are small objects and need more resolution to detect
    epochs=50,
    save_period=5,      # saves a checkpoint every x epochs
    project="/content/drive/MyDrive/runs/EXP",  # persists even if the VM resets
    patience=8,
    batch=32,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.001,
    cos_lr=True,
    weight_decay=0.0006,
    label_smoothing=0.2,  # note: has little effect on detection loss vs. classification, harmless to keep
    dropout=0.15,          # note: dropout only applies to classification heads, no-op here, harmless to keep
    name="Baseline-EXP",
    device=0
)


# Menampilkan Grafik Hasil Training

In [ ]:
print("\n--- GRAFIK PERFORMA MODEL ---")
folder_hasil = "/content/drive/MyDrive/runs/EXP/Baseline-EXP"
path_grafik = os.path.join(folder_hasil, "results.png")

if os.path.exists(path_grafik):
    display(Image(filename=path_grafik, width=800))
else:
    print("Grafik performa tidak ditemukan.")

#Custom Graph

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_dir = "/content/drive/MyDrive/runs/EXP/Baseline-EXP"
df = pd.read_csv(f"{results_dir}/results.csv")
df.columns = df.columns.str.strip()  # Ultralytics CSV headers often have leading spaces

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

train_loss = df['train/box_loss'] + df['train/cls_loss'] + df['train/dfl_loss']
axes[0].plot(df['epoch'], train_loss, label='Train Loss', linewidth=2.5, color='#e74c3c')
if all(c in df.columns for c in ['val/box_loss', 'val/cls_loss', 'val/dfl_loss']):
    val_loss = df['val/box_loss'] + df['val/cls_loss'] + df['val/dfl_loss']
    axes[0].plot(df['epoch'], val_loss, label='Val Loss', linewidth=2.5, color='#3498db')
axes[0].set_xlabel('Epoch', fontsize=13)
axes[0].set_ylabel('Loss', fontsize=13)
axes[0].set_title('Loss (box + cls + dfl)', fontsize=15, fontweight='bold')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['epoch'], df['metrics/mAP50(B)'] * 100, label='mAP50', linewidth=2.5, color='#2ecc71')
axes[1].set_xlabel('Epoch', fontsize=13)
axes[1].set_ylabel('mAP50 (%)', fontsize=13)
axes[1].set_title('Validation mAP50', fontsize=15, fontweight='bold')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Baseline-EXP Training Results', fontsize=17, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(f"{results_dir}/results_readable.png", dpi=150, bbox_inches='tight')
plt.show()


#Confusion Matrix

In [ ]:
import os
from IPython.display import Image, display

cm_path = os.path.join(folder_hasil, "confusion_matrix_normalized.png") #confusion_matrix_normalized.png or confusion_matrix.png

if os.path.exists(cm_path):
    display(Image(filename=cm_path, width=800))
else:
    print("Confusion matrix not found at:", cm_path)

# Visualisasi Dengan Data Test 1-by-1

In [ ]:
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt

model = YOLO("/content/drive/MyDrive/runs/EXP/Baseline-EXP/weights/best.pt")

test_folder = f"{dataset_path}/test/images"  # detection test images live under test/images, not test/<class>/

for file in os.listdir(test_folder):
    if file.endswith((".jpg", ".png", ".jpeg")):
        image_path = os.path.join(test_folder, file)
        results = model(image_path)

        num_detections = len(results[0].boxes)
        annotated = results[0].plot()

        plt.figure(figsize=(6, 6))
        plt.imshow(annotated[..., ::-1])
        plt.axis("off")
        plt.title(f"{file} | {num_detections} piece(s) detected")
        plt.show()


# Visualisasi Dengan Data Test Batch

In [ ]:
model = YOLO("/content/drive/MyDrive/runs/EXP/Baseline-EXP/weights/best.pt")

metrics = model.val(
    data=f"{dataset_path}/data.yaml",
    split='test',
    project="/content/drive/MyDrive/runs/EXP",
    name="Baseline-EXP-test"
)

print(f"Test mAP50: {metrics.box.map50:.2%}")
print(f"Test mAP50-95: {metrics.box.map:.2%}")
print(f"Test Precision: {metrics.box.mp:.2%}")
print(f"Test Recall: {metrics.box.mr:.2%}")


# Coba dengan Gradio

In [ ]:
from ultralytics import YOLO
import gradio as gr

model = YOLO("/content/drive/MyDrive/runs/EXP/Baseline-EXP/weights/best.pt")

def predict(image):
    results = model(image)
    annotated = results[0].plot()
    return annotated[..., ::-1]  # BGR -> RGB for Gradio display

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=gr.Image(type="numpy"),
    title="Chess Piece Detector",
    description="Upload a chessboard photo to detect and label each piece (king, queen, bishop, knight, rook, pawn - white/black)."
)

demo.launch()


# Export Model untuk Deployment


In [ ]:
model = YOLO("/content/drive/MyDrive/runs/EXP/Baseline-EXP/weights/best.pt")

# Desktop, Server, Windows, Linux, Web, C++
model.export(format="onnx")

# Android, Raspberry Pi, Embedded, Device, IoT
model.export(format="tflite")

# Android
model.export(format="ncnn")

# iPhone, iPad, MacBook, Vision Pro, Apple Watch
model.export(format="coreml")
